In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from transformers import ModernBertConfig
from tqdm import tqdm
from sklearn.metrics import classification_report






/home/hice1/kjacob7/.conda/envs/eml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = pd.read_csv("scratch/4641_data/train_dataset.csv")
print(df_train["text_length"].max())
df_train = df_train[["text", "sentiment", "bucket"]]
df_train["sentiment"] = df_train["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_val = pd.read_csv("scratch/4641_data/val_dataset.csv")
df_val = df_val[["text", "sentiment", "bucket"]]
df_val["sentiment"] = df_val["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_test = pd.read_csv("scratch/4641_data/test_dataset.csv")
df_test = df_test[["text", "sentiment", "bucket"]]
df_test["sentiment"] = df_test["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})


12930


In [3]:
class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=3000):
        self.bodies = data["text"].values
        self.labels = data["sentiment"].values
        self.buckets = data["bucket"].values if "bucket" in data.columns else None
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.bodies)
    def __getitem__(self, idx):
        encoded = self.tokenizer(str(self.bodies[idx]), truncation=True, padding=False, max_length = self.max_length, return_tensors = "pt")
        return {"input_ids": encoded["input_ids"].squeeze(), "attention_mask": encoded["attention_mask"].squeeze(), "label": torch.tensor(self.labels[idx])}

In [4]:
num_classes = 3

In [5]:
class bertSentimentAnalyzer(nn.Module):
    def __init__ (self, num_classes=num_classes, freeze_base=True):
        super().__init__()
        self.base = bert_model
        self.dropout = nn.Dropout(0.3)
        self.linear1 = nn.Linear(self.base.config.hidden_size, num_classes)
        if freeze_base:
            for param in self.base.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = self.dropout(pooled)
        return self.linear1(pooled)

In [6]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
config = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False)


buckets = ["super_short", "short", "medium", "long"]
num_classes = 3
for bucket in buckets:
    print(f"Running bucket {bucket}")
    print()
    bucket_train = df_train[df_train["bucket"] == bucket]
    bucket_val = df_val[df_val["bucket"] == bucket]
    bucket_test = df_test[df_test["bucket"] == bucket]
    
    print(len(bucket_train))
    print(len(bucket_val))
    print(len(bucket_test))
    
    
    train_data = SentimentDataset(bucket_train, tokenizer)
    val_data = SentimentDataset(bucket_val, tokenizer)
    test_data = SentimentDataset(bucket_test, tokenizer)
    
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(val_data, batch_size=16, collate_fn=collator)
    test_loader = DataLoader(test_data, batch_size=16, collate_fn=collator)
    
    
    bert_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base", config=config)
    model = bertSentimentAnalyzer(freeze_base=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    #optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    num_epochs = 20
    training_steps = num_epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*training_steps),num_training_steps=training_steps)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #print(device)
    model.to(device)
    print(device)
    best_val_loss = float("inf")
    not_improved = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        if val_loss < best_val_loss:
            not_improved = 0
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("Saved best model!")
        elif val_loss >= best_val_loss:
            not_improved += 1
            if not_improved >= 3:
                break


    print(f"Evaluating bucket {bucket}")
    model.load_state_dict(torch.load("best_model.pt"))
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")


    print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))
    print(f"End of bucket {bucket}")
    print()


Running bucket super_short

14000
2000
4000
cuda

Epoch 1/20


Evaluating: 100%|██████████| 125/125 [00:01<00:00, 79.44it/s]


Train Loss: 1.0124 | Train Acc: 0.4819
Val Loss:   0.8813 | Val Acc:   0.6000
Saved best model!

Epoch 2/20


Evaluating: 100%|██████████| 125/125 [00:01<00:00, 79.53it/s]


Train Loss: 0.8469 | Train Acc: 0.6102
Val Loss:   0.7943 | Val Acc:   0.6540
Saved best model!

Epoch 3/20


...

Train Loss: 0.6961 | Train Acc: 0.6888
Val Loss:   0.6294 | Val Acc:   0.7213

Epoch 16/20


Evaluating: 100%|██████████| 63/63 [00:04<00:00, 14.32it/s]


Train Loss: 0.7009 | Train Acc: 0.6874
Val Loss:   0.6237 | Val Acc:   0.7443
Evaluating bucket long


Evaluating: 100%|██████████| 125/125 [00:09<00:00, 13.78it/s]


Test Loss: 0.6365 | Test Acc: 0.7314
              precision    recall  f1-score   support

    negative       0.76      0.80      0.78       733
     neutral       0.66      0.65      0.66       533
    positive       0.75      0.72      0.74       733

    accuracy                           0.73      1999
   macro avg       0.72      0.72      0.72      1999
weighted avg       0.73      0.73      0.73      1999

End of bucket long

